# Fase 2 - Preprocessing & Feature Engineering

Este notebook ya no repite la lógica: la importa de `src/`. Aquí solo contamos qué pasa y
mostramos los números. El código de verdad vive en `src/data.py` y `src/features.py`, así si
mañana cambio una regla la cambio en un sitio, no en tres.

Decisiones clave (justificadas en el EDA):
- Quitar los 1.081 duplicados antes del split.
- Crear `Hour`, `Amount_log`, `Is_night`.
- Escalado robusto ajustado solo en train, y SMOTE solo en train (nunca en test).

In [1]:
from src.config import load_config
from src import data, features

cfg = load_config()
df = data.load_raw(cfg)
df, n_dups = data.clean(df)
print(f"Filas tras quitar {n_dups:,} duplicados: {len(df):,}")

Filas tras quitar 1,081 duplicados: 283,726


In [2]:
df = features.engineer(df, cfg)
df[["Hour", "Amount_log", "Is_night", "Class"]].head()

,Hour,Amount_log,Is_night,Class
0,0.000000,5.014760,1,0
1,0.000000,1.305626,1,0
2,0.000278,5.939276,1,0
3,0.000278,4.824306,1,0
4,0.000556,4.262539,1,0


Generamos los splits. `make_splits` parte primero, luego escala (fit solo en train) y aplica SMOTE solo al train. Guarda todo en `data/splits.pkl`.

In [3]:
splits = features.make_splits(df, cfg)
import numpy as np
print("Train (con SMOTE):", splits["X_train"].shape,
      "| fraude:", int(np.sum(splits["y_train"])))
print("Test  (real):     ", splits["X_test"].shape,
      "| fraude:", int(np.sum(splits["y_test"])))
print("Features:", len(splits["features"]))

Train (con SMOTE): (453204, 31) | fraude: 226602
Test  (real):      (56746, 31) | fraude: 95
Features: 31


Listo: `data/splits.pkl` queda con el train balanceado, el test intacto y el train sin SMOTE (este último lo usa la validación cruzada del notebook 03).